Run these examples from the complete `tcia_utils_v4` folder; see [setup and preview-release instructions](README.md).

# Discover a TCIA dataset and cite it

This notebook uses the built `tcia_utils` package. The default mode is a **synthetic fixture**; set `TCIA_EXAMPLE_MODE=live` to make bounded metadata calls to a TCIA V2 REST host. DataCite records enrich visible TCIA dataset records and do not establish publication alone. No imaging payload is requested.

In [ ]:
from importlib.metadata import version
from _support import MODE, SHORT_TITLE, DOI, make_client, checked
print("SYNTHETIC FIXTURE MODE — results below are examples, not live TCIA evidence" if MODE == "fixture"
      else "LIVE METADATA MODE — bounded calls to the configured TCIA service")
print("tcia_utils", version("tcia_utils"))

In [ ]:
with make_client() as tcia:
    status = checked("readiness", tcia.status)
    if status:
        print("release", status.provenance.release_fingerprint, "profile", status.data.installed_profile)
    page = checked("dataset search", lambda: tcia.search_datasets(short_titles=[SHORT_TITLE], limit=5))
    if page:
        for row in page.items:
            print("visible TCIA dataset", row.get("dataset_type"), row.get("short_title"), row.get("doi"))
        print("page", page.count, "completeness", page.completeness.state)
    detail = checked("dataset detail", lambda: tcia.get_dataset(SHORT_TITLE))
    if detail:
        print("dataset candidates", [(row.get("dataset_type"), row.get("short_title")) for row in detail.data["datasets"]])
        print("download access", [(row.get("download_id"), row.get("access_level")) for row in detail.data["current_downloads"]])
        print("related analyses are bounded suggestions", len(detail.data["related_analysis_results"]))

A short title may identify both a Collection and an Analysis Result. Keep `dataset_type` with `short_title`. Use download-level access and license evidence before making a selection.

In [ ]:
with make_client() as tcia:
    citation = checked("citation", lambda: tcia.get_citation(DOI, style="apa"))
    if citation:
        print("DataCite citation", citation.data["record"]["formatted_citation"])
        print("scope", citation.notes[0])
    impact = checked("impact", lambda: tcia.get_impact(SHORT_TITLE, publication_limit=3))
    if impact:
        for row in impact.data["impacts"]:
            print("DataCite citation events", row.get("datacite_citation_count"),
                  "verified analytical publications", row.get("verified_analytical_publication_count"))
        print("publication sublist completeness", impact.completeness.state)